# Notebook 7 — Engineering check (frozen ensemble + matrix split)

⚠️ **Statistics caveat:** Stage **8C** in `scripts/train.py` retrains each base learner on **all rows** (`final_*.pkl`). Any metric you compute below on rows drawn from **`X_combined.npz`** is therefore **optimistic** (not a clean unbiased test). Use this notebook to confirm **dimensions, softmax outputs, and code parity** — not publication-grade benchmarks.

---

We rebuild the same **stratified train/test matrices** used in **`scripts/train.py` / Notebook 04** (`test_size=0.2`, `random_state=42`, `stratify=y`). For unbiased evaluation later, freeze a CSV that never participates in Stage 8C retraining.

**Heavy alternative:** per-row `build_feature_vector` over raw CSV text — see Notebook **05**.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, classification_report, f1_score, log_loss
from sklearn.model_selection import train_test_split

ART = Path("../artifacts")
MODEL = Path("../model")

X = sp.load_npz(ART / "X_combined.npz")
y = np.load(ART / "y_labels.npy")

scaler = joblib.load(ART / "scaler.pkl")
n_dense = int(scaler.n_features_in_)
n_tfidf = X.shape[1] - n_dense
X_tfidf = X[:, :n_tfidf]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

Xt_train, Xt_test, _, _ = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y,
)

model_sgd = joblib.load(MODEL / "final_sgd.pkl")
model_nb = joblib.load(MODEL / "final_nb.pkl")
model_lr = joblib.load(MODEL / "final_logreg.pkl")
w = np.load(MODEL / "ensemble_weights.npy")

prob_sgd = model_sgd.predict_proba(X_test)
prob_nb = model_nb.predict_proba(Xt_test)
prob_lr = model_lr.predict_proba(X_test)
prob_e = (
    float(w[0]) * prob_sgd
    + float(w[1]) * prob_nb
    + float(w[2]) * prob_lr
)

preds = np.argmax(prob_e, axis=1)
targets = ["model_a_wins", "model_b_wins", "tie"]

print(f"Held-out rows: {X_test.shape[0]:,} | features={X_test.shape[1]}")
print(
    "log_loss:",
    log_loss(y_test, prob_e),
    "macro-F1:",
    f1_score(y_test, preds, average="macro"),
    "accuracy:",
    accuracy_score(y_test, preds),
)
print("\nclassification report (ensemble):")
print(classification_report(y_test, preds, target_names=targets))

meta_path = MODEL / "model_metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print(
        "\nnotebook vs metadata eval_log_loss:",
        "{:.4f} vs {}".format(
            log_loss(y_test, prob_e),
            meta.get("eval_log_loss"),
        )
    )